In [ ]:
# Setup: in Google Colab, download and install the repository (a few minutes, once per session); elsewhere, work from the
# repository. Pages that read the derived catalogues need Zenodo bundle 1 (python scripts/fetch_release.py).
import os, subprocess, sys
os.environ['MPLBACKEND'] = 'Agg'   # as in a plain script run: Jupyter's inline settings would change the saved figures
if 'google.colab' in sys.modules:
    if not os.path.exists('/content/lmc-ancient-disc'):
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/vasilybelokurov/lmc-ancient-disc', '/content/lmc-ancient-disc'], check=True)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '/content/lmc-ancient-disc'], check=True)
    os.chdir('/content/lmc-ancient-disc')
else:
    while not os.path.exists('pyproject.toml'):
        os.chdir('..')
sys.path[:0] = ['src', 'scripts', 'kb']
if not os.path.exists('data/giants_apogee.parquet'):   # the catalogues of Zenodo bundle 1, checked against data/sources.yaml
    subprocess.run([sys.executable, 'scripts/fetch_release.py'], check=True)

# Rotation and the circular speed

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasilybelokurov/lmc-ancient-disc/blob/main/kb/notebooks/chapters/kinematics_vc.ipynb)

*Second page of the kinematics chapter, after 'Rotational support, centre and sensitivity'.* Every number on this page is written by `scripts/paper_text_vc_drift.py` from the tables of the
shared-$v_c$ + drift model (`results/macros_vcdrift.tex`); the code cells are `scripts/fig_kin_vc_drift.py`, step by step, and the two scripts that make its fits, shown but not run.

The rotation $v_{\rm rot}$ of the kinematic model is the mean azimuthal velocity of a population of stars, not the circular speed
$v_c$ of the potential ($v_c^2=R\,\partial\Phi/\partial R$ in the disc plane, with $\Phi$ the gravitational potential and $R$ the
cylindrical radius in the disc frame). A population with velocity dispersion rotates more slowly than the circular speed, by its asymmetric drift,
which grows with its dispersion. The LMC's red giants, divided into five bins of iron abundance [Fe/H], rotate at 57.9–77.0 km/s at $R$ = 4 kpc, faster for the cooler,
more metal-rich bins. How much of that rise is asymmetric drift, rather than a difference in the potential the populations feel?

None of the Letter's conclusions depends on the answer: the Letter compares the rotation and dispersion of the LMC's stars with those
of simulated analogues observed and fitted in the same way, so any asymmetric drift is present in both, and it does not claim a
circular speed.

## One circular speed for all populations

We fitted all five giant bins jointly with one circular-speed curve $v_c(R)$ (the form of the free rotation curves), and let each
bin's mean rotation lag it by its asymmetric drift from the radial Jeans equation for a steady axisymmetric disc,

$$
\bar v_\phi^2 = v_c^2 - \left[\sigma_\phi^2 + \sigma_R^2\left(\frac{R}{h} - 1 - 2\,\frac{d\ln\sigma}{dR}\,R\right)\right],
$$

where $\bar v_\phi$ is the bin's mean azimuthal velocity (its rotation), $\sigma_R$ and $\sigma_\phi$ are its radial and azimuthal
velocity dispersions, $\sigma$ their common radial profile, all from the bin's own fit, and $h$ is the exponential scale length of its
surface density (the radial Jeans equation of [Binney & Tremaine 2008](https://ui.adsabs.harvard.edu/abs/2008gady.book.....B), for an exponential disc
whose velocity ellipsoid is aligned with the cylindrical axes). $h$ is measured from the bin's surface density of giants at 2–8 kpc (`lmcold.kinfit.rotation`, `scripts/kin_vc_drift.py`). On mocks with a known curve
($v_c(4\,{\rm kpc})$ = 81.9 km/s) the fit recovers $v_c$ to 0.2 km/s.

For the LMC this gives $v_c(4\,{\rm kpc})$ = 87.2 km/s. The bins then lag the circular speed by 25 km/s (the most
metal-poor) to 11 km/s (the most metal-rich): most of the rise of the rotation with metallicity is the asymmetric drift of
populations whose dispersions fall. The model predicts the RR Lyrae, which it did not fit, to rotate at 67.0 km/s (16th–84th
percentiles 61.1–72.2), against the measured 63.1 (60.8–65.4). Figure 7.8 shows the fitted curves and this comparison.

### The joint fits

`scripts/kin_vc_drift.py` makes the fits behind these numbers in four steps. It measures the scale length $h$ of each giant bin from
the selection-weighted surface density of the BOSS giants, and that of the RR Lyrae, against $R$ in the disc frame of the joint
ellipsoid fit. It then fits mocks of three populations that share one known $v_c(R)$ and rotate at their exact drift means, with
the shared-$v_c$ + drift model, with the same model and $h$ free, and with free rotation curves; this is the recovery test quoted
above. It fits the five giant bins jointly with free rotation curves and with the shared-$v_c$ + drift model on the same stars, and
finally predicts the rotation of the RR Lyrae from the fitted $v_c$ and their own dispersions and $h$, posterior draw by posterior
draw. Its tables (`results/kin_vc_drift_h.csv`, `_mock.csv`, `_data.csv`, `_rrl.csv`) are read by `scripts/paper_text_vc_drift.py`
and by the figure script.

<details><summary>About <code>scripts/kin_vc_drift.py</code></summary>

```text
The shared-v_c + drift model: one fitted circular-speed curve v_c(R) for all giant groups, each group's mean rotation lagging it
by its asymmetric drift, computed from the group's own dispersions and density scale length with the radial Jeans equation
(lmcold.kinfit.rotation with h > 0). v_c(R) is a parametrised curve fitted to the kinematics, not derived from a mass model.

Steps
1. Density scale lengths h (fixed in the fits): for each giant [Fe/H] bin (measured [Fe/H], lmcold.samples.FEH_BINS) the
   selection-weighted surface density of the BOSS giants (weight w x coverage fill x (1 - Milky Way fraction), as in
   scripts/paper_feh_profile.py) in 1-kpc annuli of in-plane radius R, with R in the frame of the joint ellipsoid fit
   (results/kin_mcmc_data_ellipsoid.csv, stage 2); an exponential (straight line in ln Sigma, unweighted) over H_FIT kpc; errors from
   NBOOT bootstraps of nside-64 sky pixels. The RR Lyrae members the same way (weight 1). The Jeans equation holds for any tracer
   selected independently of velocity, so each measured bin uses its own density, with no deconvolution of [Fe/H] errors.
2. Mocks (MOCK_GROUPS, sharing MOCK_VC): thin exponential discs of scale length h, in the joint-fit orientation, stars at
   2 < rho < 9 deg, velocities from the aligned ellipsoid with sigma_i exp(gamma (R - 4 kpc)) about the exact drift mean
   (no floor: every star must have vbar^2 > 0), proper-motion errors of the nearest real giant (Gaia covariance), line-of-sight
   errors E_VLOS; NREAL realisations. Joint maximum-likelihood fits (geometry, systemic motion and streaming shared and free):
   'drift' (one curve v0, R0, vslope shared; h fixed at the truth), 'drift, h free' and 'free' (own v0, R0, vslope per group).
3. Data: the five giant bins at 2 < rho < 9 deg (as fit_kinematics_mcmc.py), joint maximum likelihood of 'free' and 'drift' (h from
   step 1) on the same stars, starting from the stage-2 ellipsoid medians; the number of stars with vbar^2 <= 0 is reported.
4. RR Lyrae (not fitted): their mean rotation predicted from the data v_c(R) and their own ellipsoid (stage 1 of the 4-9 deg run,
   draw by draw) and h, against their measured v_rot(4 kpc).

Joint fits are restarted from their own optimum (RESTARTS) until -lnL improves by less than 0.01.

Writes results/kin_vc_drift_h.csv (population, h, h_p16, h_p84 [kpc]), results/kin_vc_drift_mock.csv (realisation, model, nll, v_c(4),
V_c, R_c, s_c, and per group vrot4 and sigma_R, sigma_phi, gamma), results/kin_vc_drift_data.csv (model, nll, n_neg, per group
vrot4 free and drift-predicted, sigma_R, sigma_phi, sigma_z, gamma, k, and the shared v_c parameters) , results/kin_vc_drift_rrl.csv (RR Lyrae v_rot(4 kpc) predicted and measured: p16, median, p84); the figures
are made by scripts/fig_kin_vc_drift.py and the numbers of the KB page 'Rotation and the circular speed' by
scripts/paper_text_vc_drift.py. Diagnostic (not in the Letter). Its joint_params() is also imported by scripts/sims_observe.py,
and its fit helpers by scripts/kin_vc_drift_extensions.py.

Inputs: the release giants (lmcold.samples.giants(), Zenodo bundle 1); the BOSS mass sample (lmcold.mass_samples.samples) with
data/giant_foreground.parquet (coverage fill and Milky Way fraction); data/rrab.parquet (RR Lyrae members); the joint ellipsoid
fit results/kin_mcmc_data_ellipsoid.csv and the posterior draws data/kin_mcmc_data_ellipsoid_outer_samples.npz
(scripts/fit_kinematics_mcmc.py).

Run: .venv/bin/python scripts/kin_vc_drift.py   (run time not recorded)
```
</details>

*Shown, not run here: the release catalogues, the joint ellipsoid fit and joint maximum-likelihood fits of mocks and of the five giant bins. The cells below are the script itself; run it from the repository with* `python scripts/kin_vc_drift.py`*.*

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

#### Settings
The annuli and radial range of the scale-length fits, the parameters shared or fitted per group in the joint fits, and the
circular speed and groups of the mocks.

```python
import healpy as hp
import jax
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.spatial import cKDTree

from fit_kinematics_mcmc import COLS, ring
from lmcold import geometry as geo
from lmcold import kinfit, samples
from lmcold.constants import KAPPA, LMC
from lmcold.footprint import angular_distance
from lmcold.mass_samples import samples as mass_samples
from lmcold.sources import DATA, ROOT

P = kinfit.PARAMS.index  # P(name): position of a parameter in a kinfit.PARAMS vector
R_EDGES = np.arange(0., 11., 1.)  # kpc, annuli of the surface densities
H_FIT = (2., 8.)  # kpc, range of the exponential fits (inside 2 kpc the bar; RR Lyrae cover all azimuths to 8 kpc)
NBOOT = 100  # bootstraps of the sky pixels for the errors of h
SHARED_FREE = ('dx', 'dy', 'pmra0', 'pmdec0', 'inc', 'theta', 'vlos0') + kinfit.STREAM  # free and shared in every joint fit
CURVE = ('v0', 'R0', 'vslope')  # the rotation curve: shared (v_c) in 'drift', own in 'free'
OWN = kinfit.ELLIPSOID + ('k', 'logit_fout')  # per group always: sigma_R, sigma_phi, sigma_z [km/s], dlnsig_dR [1/kpc], error scale, outliers
MOCK_VC = dict(v0=85.0, R0=2.0, vslope=-0.02)  # mock circular speed: v0 [km/s], R0 [kpc], vslope [1/kpc]
# Mock groups, hot to cold: ellipsoid at R_SIG [km/s], its radial gradient dlnsig_dR [1/kpc], density scale length h [kpc].
MOCK_GROUPS = [dict(sigma_R=38., sigma_phi=31., sigma_z=32., dlnsig_dR=-0.05, h=2.0),
               dict(sigma_R=30., sigma_phi=25., sigma_z=24., dlnsig_dR=-0.07, h=1.6),
               dict(sigma_R=21., sigma_phi=17., sigma_z=16., dlnsig_dR=-0.08, h=1.2)]
N_MOCK, NREAL, E_VLOS = 4000, 3, 4.0  # stars per mock group, realisations, line-of-sight velocity error [km/s]
RESTARTS = 5  # restarts of a joint fit from its own optimum, until -lnL changes by < 0.01
rng = np.random.default_rng(7)  # used by the bootstraps of scale_length
```

#### Helpers
Reading the joint-fit frame, radii in the disc plane, the scale-length fit, the joint maximum-likelihood fit of several groups,
and small functions for the drift model and the mocks.

```python
def joint_params(csv='kin_mcmc_data_ellipsoid.csv'):
    """Stage-2 posterior medians of results/<csv>: dict population -> kinfit.PARAMS vector (shared plus own)."""
    # stage 2 = the joint fit: rows of the shared parameters (population 'shared') and of each population's own parameters
    t = pd.read_csv(ROOT / 'results' / csv)
    t = t[t.stage == 2]
    out = {}
    # one vector per population: the shared values plus its own, other parameters at kinfit.start()
    for pop in t.population.unique():
        if pop == 'shared':
            continue
        p = kinfit.start()
        for _, r in t[t.population.isin(['shared', pop])].iterrows():
            p[P(r.param)] = r['median']
        out[pop] = p
    return out


def radius(p, ra, dec):
    """In-plane radius R [kpc] of sky positions in the frame of parameter vector p.

    p : kinfit.PARAMS vector (centre, inc, theta used); ra, dec : arrays of shape (N,) [deg]. Returns R, shape (N,).
    """
    # kinfit.data needs proper motions and errors; dummy values (zeros, unit errors) suffice for the geometry
    n = len(ra)
    d = kinfit.data(ra, dec, np.zeros(n), np.zeros(n), np.ones(n), np.ones(n), np.zeros(n))
    # element 4 of kinfit._model's output: in-plane radius R [kpc] of each star
    return np.asarray(kinfit._model(p, d['rhat'], d['ea'], d['ed'])[4])


def scale_length(R, w, pix):
    """Exponential scale length [kpc] of the weighted surface density over H_FIT, and its 16th, 84th bootstrap percentiles.

    R : in-plane radii [kpc], w : weights (selection-weighted counts), pix : HEALPix pixel of each star (the bootstrap resamples
    whole pixels); all shape (N,). Returns (h, h_p16, h_p84) [kpc]: h = -1 / slope of ln Sigma against R in the annuli R_EDGES.
    """
    # annulus centres [kpc] and areas [kpc^2]; m: annuli inside H_FIT
    mid, area = .5 * (R_EDGES[1:] + R_EDGES[:-1]), np.pi * np.diff(R_EDGES ** 2)
    m = (mid > H_FIT[0]) & (mid < H_FIT[1])

    def h(weights):
        # weighted surface density per annulus [weight / kpc^2]
        sigma = np.histogram(R, R_EDGES, weights=weights)[0] / area
        k = m & (sigma > 0)  # annuli with stars (a compact simulated population can leave the outer ones empty)
        if k.sum() < 2:
            return np.nan
        # h = -1 / slope of the straight-line fit of ln Sigma against R
        return -1 / np.polyfit(mid[k], np.log(sigma[k]), 1)[0]

    # bootstrap over HEALPix pixels: weights times each pixel's multiplicity in a resampling of the pixels with replacement
    upix, inv = np.unique(pix, return_inverse=True)
    boot = [h(w * np.bincount(rng.integers(0, len(upix), len(upix)), minlength=len(upix))[inv]) for _ in range(NBOOT)]
    return h(w), *np.percentile(boot, [16, 84])


def joint_fit(ds, p0s, shared, extra=(), own=CURVE + OWN, add=None):
    """Joint maximum likelihood (ellipsoid likelihood) of populations ds: names in `shared` common (taken from p0s[0]), `own`
    (CURVE unless shared, OWN) and `extra` (e.g. ('h',)) per population, the rest held at p0s.

    add : (names, starts, bounds, adjust) or None
        Extra shared scalars outside PARAMS (e.g. streaming slopes, a thickness coefficient) and a function adjust(vectors, values)
        returning the population vectors they modify.

    Returns (vectors, errors (NaN: no Hessian), nll, info): info has 'extra' (the fitted add values), 'success' (L-BFGS-B
    converged on the last restart) and 'grad' (max |projected gradient| at the optimum).
    """
    # parameters fitted per population: `own` and `extra`, unless shared
    own = [n for n in tuple(own) + tuple(extra) if n not in shared]
    # whether each population has line-of-sight velocities; vlos0 is fitted only if one has
    has_los = [bool(np.any(np.asarray(d['los']))) for d in ds]
    sh = [n for n in shared if n != 'vlos0' or any(has_los)]
    # positions of the shared and own parameters in a kinfit.PARAMS vector
    idx_sh, idx_own = np.array([P(n) for n in sh], dtype=int), np.array([P(n) for n in own], dtype=int)
    names, starts, add_bounds, adjust = add if add else ((), (), (), None)
    n_add = len(names)
    # x0: [shared values from p0s[0], own values of each population in turn, extra shared scalars]
    x0 = np.concatenate([p0s[0][idx_sh]] + [p[idx_own] for p in p0s] + [np.asarray(starts, float)])
    # bounds in the same order; the start is moved just inside them
    bounds = [kinfit.BOUNDS[n] for n in sh] + [kinfit.BOUNDS[n] for n in own] * len(ds) + list(add_bounds)
    x0 = np.clip(x0, [b[0] + 1e-6 for b in bounds], [b[1] - 1e-6 for b in bounds])

    def vectors(x):
        # vectors(x): the full kinfit vector of each population from the packed parameter array x
        out = []
        for i, p in enumerate(p0s):
            q = kinfit.jnp.asarray(p).at[idx_sh].set(x[:len(sh)])
            out.append(q.at[idx_own].set(x[len(sh) + i * len(own): len(sh) + (i + 1) * len(own)]))
        return adjust(out, x[len(x) - n_add:]) if n_add else out

    # f: total negative log-likelihood of all populations (ellipsoid likelihood), jit-compiled; g: its gradient
    f = jax.jit(lambda x: sum(kinfit.neg_loglike(q, d, True) for q, d in zip(vectors(x), ds)))
    g = jax.jit(jax.grad(f))
    x, last = x0, np.inf
    for _ in range(RESTARTS):  # L-BFGS-B can stop at its iteration limit on these large joint fits: restart until it stalls
        r = minimize(lambda q: float(f(q)), x, jac=lambda q: np.asarray(g(q)), method='L-BFGS-B', bounds=bounds,
                     options=dict(ftol=1e-15, gtol=1e-6, maxiter=20000, maxfun=50000))
        if r.fun < last:  # keep only improvements
            x = r.x
        if last - r.fun < 0.01:
            break
        last = min(last, r.fun)
    # No Hessian: for ~57 parameters over ~57 000 stars it exhausts memory (the process aborts); errors come from NUTS.
    # projected gradient at the optimum: components at a bound pointing outward set to zero
    lo, hi = np.array(bounds).T
    gr = np.asarray(g(x))
    proj = np.where((x <= lo + 1e-9) & (gr > 0) | (x >= hi - 1e-9) & (gr < 0), 0.0, gr)  # gradient pointing out of the box ignored
    info = dict(extra=np.asarray(x[len(x) - n_add:]) if n_add else np.array([]), success=bool(r.success), grad=float(np.abs(proj).max()))
    # the population vectors at the optimum; errors returned as NaN (see above)
    ps = [np.array(q) for q in vectors(x)]
    return ps, [np.full(len(kinfit.PARAMS), np.nan)] * len(ds), float(f(x)), info


def n_negative(p, d):
    """Stars whose asymmetric-drift mean has vbar^2 <= 0 (no equilibrium) for vector p (h > 0)."""
    # in-plane radius R [kpc] of each star
    R = np.asarray(kinfit._model(p, d['rhat'], d['ea'], d['ed'])[4])
    # circular speed v_c(R) [km/s] of the parametrised curve (tanh rise times a linear slope about R_SIG)
    vc = p[P('v0')] * np.tanh(R / p[P('R0')]) * (1 + p[P('vslope')] * (R - kinfit.R_SIG))
    # g: radial gradient of ln sigma [1/kpc]
    g = p[P('dlnsig_dR')]
    # u2 = vbar^2 from the radial Jeans equation for an exponential density of scale length h and dispersions exp(g (R - R_SIG))
    u2 = vc ** 2 - np.exp(2 * g * (R - kinfit.R_SIG)) * (p[P('sigma_phi')] ** 2 + p[P('sigma_R')] ** 2 * (R / p[P('h')] - 1 - 2 * g * R))
    return int((u2 <= 0).sum())


def vrot4(p):
    """Mean rotation [km/s] at R_SIG of vector p (free curve, or drift mean when h > 0). With np.r_[p[:32], 0.0] (h set to 0),
    the circular speed v_c(R_SIG) of the vector."""
    return float(kinfit.rotation(p, np.array([kinfit.R_SIG]))[0])


def mock(group, frame, err_cat, seed):
    """Catalogue dict of one mock group (module docstring, step 2) in the frame (centre, orientation, systemic motion) of `frame`,
    and its true parameter vector."""
    # own random generator per mock group (seed passed in)
    g = np.random.default_rng(seed)
    p = frame.copy()
    # the frame's centre, orientation and systemic motion, with the mock curve and the group's ellipsoid and h
    for k, v in (MOCK_VC | group).items():
        p[P(k)] = v
    # no streaming in the mocks
    p[[P(n) for n in kinfit.STREAM]] = 0.0
    # r0: unit vector to the centre; eE0, eN0: east and north unit vectors there; line: line of nodes; n: disc normal
    r0, eE0, eN0, line, n = (np.asarray(v) for v in geo.symmetry_axis(*(float(v) for v in kinfit.centre(p[0], p[1])),
                                                                       np.deg2rad(p[P('inc')]), np.deg2rad(p[P('theta')])))
    m = 30 * N_MOCK  # stars drawn before the 2 < rho < 9 deg cut keeps N_MOCK of them
    R, ph = g.gamma(2.0, group['h'], m), g.uniform(0, 2 * np.pi, m)  # exponential surface density: R ~ Gamma(2, h)
    # 3D positions [kpc]: centre at distance D0 plus (R, phi) in the disc plane
    xyz = LMC['D0'] * r0 + R[:, None] * (np.cos(ph)[:, None] * line + np.sin(ph)[:, None] * np.cross(n, line))
    # unit vectors to the stars, then (ra, dec) [deg] and distance from the vdMK14 centre rho [deg]
    rhat = xyz / np.linalg.norm(xyz, axis=1)[:, None]
    ra, dec = np.rad2deg(np.arctan2(rhat[:, 1], rhat[:, 0])) % 360, np.rad2deg(np.arcsin(rhat[:, 2]))
    rho = angular_distance(ra, dec, LMC['ra0'], LMC['dec0'])
    keep = np.flatnonzero((rho > 2) & (rho < 9))[:N_MOCK]  # the ring of fit_kinematics_mcmc.RING about the vdMK14 centre
    ra, dec = ra[keep], dec[keep]
    # unit vectors of the real giants; each mock star takes the proper-motion errors of the nearest one
    xyzc = np.c_[np.cos(np.deg2rad(err_cat['dec'])) * np.cos(np.deg2rad(err_cat['ra'])),
                 np.cos(np.deg2rad(err_cat['dec'])) * np.sin(np.deg2rad(err_cat['ra'])), np.sin(np.deg2rad(err_cat['dec']))]
    near = cKDTree(xyzc).query(rhat[keep])[1]
    e = {k: np.asarray(err_cat[k])[near] for k in ('pmra_error', 'pmdec_error', 'pmra_pmdec_corr')}
    # the kinfit data of the mock stars (proper motions zero for now: only the geometry and errors are used)
    d = kinfit.data(ra, dec, np.zeros(len(ra)), np.zeros(len(ra)), e['pmra_error'], e['pmdec_error'], e['pmra_pmdec_corr'])
    # D: distance [kpc]; Rm: in-plane radius [kpc]; eR, ephi, nn: radial, azimuthal and normal unit vectors (N, 3)
    _, _, _, D, Rm, eR, ephi, nn = (np.asarray(v) for v in kinfit._model(p, d['rhat'], d['ea'], d['ed']))
    assert n_negative(p, d) == 0, 'mock with vbar^2 <= 0'
    # dispersion scale factor exp(dlnsig_dR (R - R_SIG)) per star
    scale = np.exp(p[P('dlnsig_dR')] * (Rm - kinfit.R_SIG))
    vbar = np.asarray(kinfit.rotation(p, Rm))  # exact drift mean (no floor reached: n_negative == 0 and u2 >> DRIFT_EPS^2)
    # 3D velocities (N, 3) [km/s]: systemic motion of the centre plus random radial, azimuthal (about vbar) and vertical parts
    V = (p[P('vlos0')] * r0 + KAPPA * LMC['D0'] * (p[P('pmra0')] * eE0 + p[P('pmdec0')] * eN0))[None] \
        + g.normal(0, group['sigma_R'] * scale)[:, None] * eR + g.normal(vbar, group['sigma_phi'] * scale)[:, None] * ephi \
        + g.normal(0, group['sigma_z'] * scale)[:, None] * nn
    # proper motions [mas/yr]: velocity projected on east/north / (KAPPA x D), plus Gaia noise (Cholesky of the covariance)
    ea, ed, rh = (np.asarray(d[k]) for k in ('ea', 'ed', 'rhat'))
    noise = np.einsum('nij,nj->ni', np.linalg.cholesky(np.asarray(d['cov'])), g.normal(size=(len(ra), 2)))
    pmra, pmdec = (V * ea).sum(1) / (KAPPA * D) + noise[:, 0], (V * ed).sum(1) / (KAPPA * D) + noise[:, 1]
    # line-of-sight velocity [km/s] plus Gaussian error E_VLOS
    vlos = (V * rh).sum(1) + g.normal(0, E_VLOS, len(ra))
    # no distance moduli in the mocks (dmu NaN)
    return dict(ra=ra, dec=dec, pmra=pmra, pmdec=pmdec, vlos=vlos, evlos=np.full(len(ra), E_VLOS),
                dmu=np.full(len(ra), np.nan), **e), p


def summary(model, ps, es, nll, names, ds, extra=None):
    """Rows for the csv: per population its curve, v_rot(4 kpc) (drift mean when h > 0), v_c(4 kpc), ellipsoid, k, f, h, and the
    number of its stars with vbar^2 <= 0."""
    rows = []
    # one row per population: rotation at 4 kpc, circular speed at 4 kpc (h set to 0), h, stars with vbar^2 <= 0
    for name, p, e, d in zip(names, ps, es, ds):
        r = dict(model=model, population=name, nll=nll, vrot4=vrot4(p), vc4=vrot4(np.r_[p[:32], 0.0]), h=p[P('h')],
                 h_err=e[P('h')], n_neg=n_negative(p, d) if p[P('h')] > 0 else 0)
        # plus the curve, ellipsoid, error scale, outlier fraction, inc and theta, and their errors
        r |= {n: p[P(n)] for n in CURVE + OWN + ('inc', 'theta')} | {f'{n}_err': e[P(n)] for n in CURVE + OWN}
        rows.append(r | (extra or {}))
    return rows


def with_h(ps, hs):
    """Copies of the vectors ps with h set to hs (0 switches the drift off)."""
    out = []
    for p, h in zip(ps, hs):  # hs [kpc]
        q = p.copy()
        q[P('h')] = h
        out.append(q)
    return out
```

#### Load the data
The disc frame of the joint ellipsoid fit (stage-2 medians per population) and the giants of the five [Fe/H] bins at
2 < $\rho$ < 9 deg (no distance moduli: NaN). The BOSS mass sample, the foreground table and the RR Lyrae catalogue are read in
the next step, where the scale lengths are measured, and the RR Lyrae posterior draws in the last step.

```python
frames = joint_params()  # population -> kinfit.PARAMS vector (results/kin_mcmc_data_ellipsoid.csv)
# giants of the five bins, cut to 2 < rho < 9 deg; dmu NaN (distance moduli not used)
giants = [(f'giants {lab}', ring(cat | dict(dmu=np.full(len(cat['ra']), np.nan)))) for lab, cat in samples.giants()]
# population labels, e.g. 'giants -0.8 < [Fe/H] < -0.5'
names = [n for n, _ in giants]
frame = frames[names[0]].copy()  # its shared part is the joint-fit frame
```

#### Density scale lengths
The exponential scale length $h$ of each giant bin, from the selection-weighted surface density of the BOSS giants of the mass
sample, and of the RR Lyrae members, both against the in-plane radius $R$ of the joint-fit frame; errors from bootstraps of
sky pixels. Writes `results/kin_vc_drift_h.csv`.

```python
b, _, _ = mass_samples()  # BOSS mass sample: one row per giant, with selection weight w and [Fe/H] feh
# coverage fill and Milky Way fraction per star (columns fill, f_mw)
fg = pd.read_parquet(DATA / 'giant_foreground.parquet')
b = b.merge(fg[['source_id', 'fill', 'f_mw']], on='source_id', validate='one_to_one')
b['W'] = b.w * b['fill'] * (1 - b.f_mw)  # weight: selection x coverage fill x (1 - Milky Way fraction), as paper_feh_profile.py
b['R'] = radius(frame, b.ra.to_numpy(), b.dec.to_numpy())  # kpc
# nside-64 HEALPix pixel of each star (the bootstrap unit)
b['pix'] = hp.ang2pix(64, b.ra.to_numpy(), b.dec.to_numpy(), lonlat=True)
hrows = []
# one scale length per measured [Fe/H] bin (samples.FEH_BINS) [dex]
for (lo, hi), name in zip(samples.FEH_BINS, names):
    s = b[(b.feh >= lo) & (b.feh < hi)]
    hrows.append(dict(population=name, **dict(zip(('h', 'h_p16', 'h_p84'),
                                                   scale_length(s.R.to_numpy(), s.W.to_numpy(), s.pix.to_numpy())))))
# the RR Lyrae members, weight 1
rr = pd.read_parquet(DATA / 'rrab.parquet')
rr = rr[rr.member]
hrows.append(dict(population='RR Lyrae', **dict(zip(('h', 'h_p16', 'h_p84'), scale_length(
    radius(frame, rr.ra.to_numpy(), rr.dec.to_numpy()), np.ones(len(rr)),
    hp.ang2pix(64, rr.ra.to_numpy(), rr.dec.to_numpy(), lonlat=True))))))
# population -> h, h_p16, h_p84 [kpc]
H = pd.DataFrame(hrows).set_index('population')
H.to_csv(ROOT / 'results/kin_vc_drift_h.csv', float_format='%.3f')
print(H.round(2).to_string(), flush=True)
```

#### Mocks with a known circular speed
Three mock groups share one $v_c(R)$ and rotate at their exact drift means; each realisation is fitted with 'drift' ($h$ at
the truth), 'drift, h free' and 'free'. Writes `results/kin_vc_drift_mock.csv`.

```python
# Positions and Gaia proper-motion errors of all giants: each mock star takes the errors of the nearest one.
err_cat = {k: np.concatenate([c[k] for _, c in giants]) for k in ('ra', 'dec', 'pmra_error', 'pmdec_error', 'pmra_pmdec_corr')}
mrows = []
# NREAL realisations of the three mock groups; seeds 100 x realisation + group
for real in range(NREAL):
    cats, truths = zip(*(mock(g, frame, err_cat, 100 * real + j) for j, g in enumerate(MOCK_GROUPS)))
    # kinfit data of each mock catalogue
    ds = [kinfit.data(*(c[k] for k in COLS)) for c in cats]
    mnames = [f'mock {j}' for j in range(len(ds))]
    starts = []
    for t in truths:  # start away from the truth: v0, R0 of a plain tanh, isotropic 30 km/s, no gradient
        q = t.copy()
        q[[P(n) for n in ('v0', 'R0', 'vslope', 'sigma_R', 'sigma_phi', 'sigma_z', 'dlnsig_dR')]] = [70., 2.5, 0., 30., 30., 30., 0.]
        starts.append(q)
    # true scale lengths of the mock groups [kpc]
    hs = [t[P('h')] for t in truths]
    # 'drift': one shared curve, h fixed at the truth
    ps, es, nll, _ = joint_fit(ds, with_h(starts, hs), SHARED_FREE + CURVE)
    mrows += summary('drift', ps, es, nll, mnames, ds, dict(realisation=real))
    # 'drift, h free': started from the 'drift' optimum, h fitted per group
    ps2, es2, nll2, _ = joint_fit(ds, ps, SHARED_FREE + CURVE, extra=('h',))
    mrows += summary('drift, h free', ps2, es2, nll2, mnames, ds, dict(realisation=real))
    # 'free': h = 0 (no drift), own curve per group
    ps3, es3, nll3, _ = joint_fit(ds, with_h(starts, [0.] * 3), SHARED_FREE)
    mrows += summary('free', ps3, es3, nll3, mnames, ds, dict(realisation=real))
    # recovered v_c(4 kpc) and drift parameters against the truth; likelihood gain of 'free' over 'drift'
    print(f'mock {real}: truth v_c(4) {vrot4(np.r_[truths[0][:32], 0.]):.1f}; drift {vrot4(np.r_[ps[0][:32], 0.]):.1f} '
          f'(V {ps[0][P("v0")]:.1f}, R0 {ps[0][P("R0")]:.2f}, s {ps[0][P("vslope")]:.3f}); h free {[round(p[P("h")], 2) for p in ps2]}; '
          f'free v_rot(4) {[round(vrot4(p), 1) for p in ps3]} vs true mean {[round(vrot4(t), 1) for t in truths]}; '
          f'dlnL drift - free {nll3 - nll:.1f}', flush=True)
pd.DataFrame(mrows).to_csv(ROOT / 'results/kin_vc_drift_mock.csv', index=False, float_format='%.4f')
```

#### The giant bins
Joint maximum-likelihood fits of the five giant bins: free rotation curves ('free'), one circular speed with each bin's drift
('drift', $h$ from the scale-length step) and the same with $h$ free. Writes `results/kin_vc_drift_data.csv`.

```python
# kinfit data of the five giant bins
ds = [kinfit.data(*(c[k] for k in COLS)) for _, c in giants]
# 'free': own curves, h = 0, started from the stage-2 medians
pf, ef, nllf, _ = joint_fit(ds, with_h([frames[n] for n in names], [0.] * 5), SHARED_FREE)
# 'drift' starts: the free solution with each bin's measured h
p0 = with_h(pf, H.loc[names, 'h'].to_numpy())
for p in p0:  # start of the common curve: v_c a little above the fastest free curve
    p[[P(n) for n in CURVE]] = [90., 2.5, 0.]
# The curve alone first, everything else at the free solution: from there the gradient of the full problem is large enough
# that the first L-BFGS-B step reaches the corners of BOUNDS, where the likelihood is not finite.
p0, _, _, _ = joint_fit(ds, p0, CURVE, own=())
# 'drift': everything shared and own free, then 'drift, h free'
pj, ej, nllj, _ = joint_fit(ds, p0, SHARED_FREE + CURVE)
ph, eh, nllh, _ = joint_fit(ds, pj, SHARED_FREE + CURVE, extra=('h',))
# one table: rows per (model, population)
rows = (summary('free', pf, ef, nllf, names, ds) + summary('drift', pj, ej, nllj, names, ds)
        + summary('drift, h free', ph, eh, nllh, names, ds))
T = pd.DataFrame(rows)
T.to_csv(ROOT / 'results/kin_vc_drift_data.csv', index=False, float_format='%.4f')
print(T[['model', 'population', 'nll', 'vrot4', 'vc4', 'h', 'h_err', 'n_neg', 'sigma_R', 'sigma_phi', 'dlnsig_dR', 'k']].round(2).to_string(index=False))
print(f'dlnL drift - free {nllf - nllj:.1f} (12 fewer parameters: one curve for five); h free - h fixed {nllj - nllh:.1f}', flush=True)
```

#### The RR Lyrae prediction
The RR Lyrae were not fitted: their mean rotation at $R$ = 4 kpc is predicted from the fitted $v_c$, their own $h$ and their
ellipsoid (stage-1 posterior draws of the 4-9 deg fit), draw by draw, and compared with their measured $v_{\rm rot}(4\,{\rm kpc})$.
Writes `results/kin_vc_drift_rrl.csv`.

```python
# stage-1 posterior samples of the 4-9 deg fit
z = np.load(DATA / 'kin_mcmc_data_ellipsoid_outer_samples.npz')
i = list(z['names']).index('RR Lyrae')
g = lambda n: z[f'1|{i}|{n}_0'].reshape(-1)  # noqa: E731  (all stage-1 draws of RR Lyrae parameter n)
# one vector per draw: data 'drift' fit of the first giant bin (indices 0-31, with the shared v_c) and the RR Lyrae h
q = np.tile(np.r_[pj[0][:32], H.loc['RR Lyrae', 'h']], (len(g('v0')), 1))
# RR Lyrae's own ellipsoid and gradient, draw by draw
for n in ('sigma_R', 'sigma_phi', 'dlnsig_dR'):
    q[:, P(n)] = g(n)
# predicted mean rotation at R_SIG = 4 kpc for each draw [km/s]
pred = np.array([vrot4(x) for x in q])
# measured rotation at R_SIG: v0 tanh(R_SIG / R0) (the vslope term vanishes at R_SIG)
meas = g('v0') * np.tanh(kinfit.R_SIG / g('R0'))
print('RR Lyrae v_rot(4 kpc): predicted', np.percentile(pred, [16, 50, 84]).round(1), 'measured', np.percentile(meas, [16, 50, 84]).round(1))
# 16th, 50th, 84th percentiles of both [km/s]
pd.DataFrame(dict(predicted=np.percentile(pred, [16, 50, 84]), measured=np.percentile(meas, [16, 50, 84])),
             index=['p16', 'median', 'p84']).to_csv(ROOT / 'results/kin_vc_drift_rrl.csv', float_format='%.2f')
```

### The figures

<details><summary>About <code>scripts/fig_kin_vc_drift.py</code></summary>

```text
Diagnostic figures of the shared-v_c + drift model (scripts/kin_vc_drift.py, scripts/kin_vc_drift_extensions.py): one fitted
circular-speed curve for the five giant [Fe/H] bins, each bin's mean rotation lagging it by its asymmetric drift (radial Jeans
equation), against free rotation curves per bin. Reads the fitted vectors (data/kin_vc_drift_ext_vectors.npz) and result tables;
fits nothing.

Data against model. Every giant with a proper motion and a line-of-sight velocity has a measured 3D velocity
V = kappa D (pmra e_E + pmdec e_N) + vlos r_hat [km/s], D its disc distance in the fit's frame. Its internal velocity in a fit p is
V - V_frame(p), where V_frame is the same fit with the rotation switched off (v0 = 0, h = 0: centre, systemic motion and
perspective only), projected on the star's e_phi (azimuthal; positive along the rotation, so v_phi > 0) and e_R. The model's
internal velocity of the star is V_model(p) - V_frame(p) (its mean rotation with the m = 2 streaming). Both are averaged over the
stars of a bin (mean, error = standard deviation / sqrt(n)); the measurement errors average out, the projection is linear.
Stars whose 3D residual from the free fit exceeds CLIP km/s are dropped (Milky Way stars, gross errors) in every model.

Writes (all diagnostic):
- figures/fig_kin_vc_drift.png: (a) the shared v_c(R) and each bin's mean rotation, free (solid) and shared-v_c + drift (dashed);
  (b) v_rot(4 kpc) shared-v_c + drift against free per bin, and the RR Lyrae prediction (results/kin_vc_drift_rrl.csv).
- figures/fig_kin_vc_drift_deficit.png: summed ln L(shared v_c + drift) - ln L(free) per bin of R (a) and azimuth (b)
  (results/kin_vc_drift_deficit.csv).
- figures/fig_kin_vc_drift_data.png: 2 x 5 panels, one column per [Fe/H] bin. Top: mean v_phi of the stars in 1-kpc bins of R
  (points, free-fit frame) and of the models on the same stars: free (grey), shared v_c + drift (black dashed), shared v_c + drift
  with population-dependent streaming and thickness (black dotted). Bottom: data - model of v_phi for free (open) and shared
  v_c + drift (filled), each in its own frame.
- figures/fig_kin_vc_drift_data_phi.png: 2 x 5 panels, data - model of v_phi (top) and v_R (bottom) in 45-deg bins of azimuth
  (from the line of nodes, along the rotation) at 2 < R < 9 kpc, free (open) and shared v_c + drift (filled).
- results/kin_vc_drift_profiles.csv: population, model, coordinate, lo, hi, n, data and
  model mean v_phi and v_R [km/s] and the error of the data mean.

Run: .venv/bin/python scripts/fig_kin_vc_drift.py   (after scripts/kin_vc_drift_extensions.py)
```
</details>

**Data read by `scripts/fig_kin_vc_drift.py`** (from the run log)

| File | Where it comes from | What it is |
|---|---|---|
| `data/giants_apogee.parquet` | Zenodo bundle 1 | APOGEE DR17 giants with Gaia DR3 |
| `data/giants_boss.parquet` | Zenodo bundle 1 | SDSS-V DR20 BOSS giants with Gaia DR3, on the APOGEE scale |
| `data/kin_vc_drift_ext_vectors.npz` | Zenodo bundle 1 | Fitted parameter vectors of the free and shared-v_c + drift joint fits of the giant bins (and their variants) |
| `results/kin_vc_drift_deficit.csv` | repository | log-likelihood deficit of the shared-v_c + drift model per bin; written by `python scripts/kin_vc_drift_extensions.py` |
| `results/kin_vc_drift_rrl.csv` | repository | RR Lyrae rotation predicted by the shared-v_c + drift model and measured; written by `python scripts/kin_vc_drift.py` |

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from fit_kinematics_mcmc import COLS, ring
from kin_vc_drift import P, vrot4
from kin_vc_drift_extensions import azimuth
from lmcold import kinfit, samples
from lmcold.constants import KAPPA
from lmcold.layout import panel_grid
from lmcold.sources import DATA, ROOT

CLIP = 150.0  # km/s, as scripts/paper_kin_residuals.py: stars with a larger 3D residual from the free fit are dropped
R_EDGES = np.arange(2., 10., 1.)  # kpc: 1-kpc bins of in-plane radius R from 2 to 9 kpc
PHI_EDGES = np.linspace(0., 360., 9)  # deg: eight 45-deg bins of azimuth from the line of nodes
COLOURS = ['#d7191c', '#fdae61', '#a6d96a', '#1a9641', '#2878b5']  # giant [Fe/H] bins, metal-poor first
# key of the fitted vectors in data/kin_vc_drift_ext_vectors.npz: legend label
MODELS = {'free': 'free', 'drift': 'shared $v_c$ + drift', 'drift_p_stream_p_beta': '+ streaming + thickness'}


def velocities(p, d):
    """Measured and model internal velocities (v_phi, v_R) [km/s] of the stars of d in the frame of p, the 3D residual
    |V - V_model| [km/s], and R [kpc], phi [deg]."""
    q = p.copy()
    q[[P('v0'), P('h')]] = 0.0  # frame only: no rotation, no drift
    # Model proper motions [mas/yr], vlos [km/s], disc distance D [kpc] and unit vectors eR, ephi (N, 3) for the full fit p.
    mra, mde, mlos, D, _, eR, ephi, _ = (np.asarray(v) for v in kinfit._model(p, d['rhat'], d['ea'], d['ed']))
    # The same for the frame-only vector q.
    fra, fde, flos = (np.asarray(v) for v in kinfit.model(q, d['rhat'], d['ea'], d['ed'])[:3])
    ea, ed, rh = (np.asarray(d[k]) for k in ('ea', 'ed', 'rhat'))
    # 3D velocity [km/s], shape (N, 3): kappa D (pmra e_E + pmdec e_N) + vlos r_hat (KAPPA = 4.74 km/s per kpc mas/yr).
    vec = lambda a, b, c: KAPPA * D[:, None] * (a[:, None] * ea + b[:, None] * ed) + c[:, None] * rh  # noqa: E731
    V, Vm, Vf = vec(np.asarray(d['pmra']), np.asarray(d['pmdec']), np.asarray(d['vlos'])), vec(mra, mde, mlos), vec(fra, fde, flos)
    R, phi = azimuth(p, d)  # in-plane radius [kpc] and azimuth [deg] in the frame of p
    # Projection on (e_phi, e_R): returns (v_phi, v_R) [km/s].
    proj = lambda X: ((X * ephi).sum(1), (X * eR).sum(1))  # noqa: E731
    return proj(V - Vf), proj(Vm - Vf), np.linalg.norm(V - Vm, axis=1), R, phi


def binned(x, edges, *ys):
    """Mean of each y in bins of x (edges: bin edges, same units as x), the standard error (sd / sqrt n) of the first, and the counts;
    bins with 4 stars or fewer give NaN."""
    k = np.digitize(x, edges) - 1  # bin index per star; -1 or len(edges) - 1 outside the edges
    # Per y: the mean in each bin with more than 4 stars.
    out = [np.array([y[k == i].mean() if (k == i).sum() > 4 else np.nan for i in range(len(edges) - 1)]) for y in ys]
    # Standard error of the mean of the first y, and the star counts per bin.
    err = np.array([ys[0][k == i].std() / np.sqrt((k == i).sum()) if (k == i).sum() > 4 else np.nan for i in range(len(edges) - 1)])
    return out, err, np.bincount(k[(k >= 0) & (k < len(edges) - 1)], minlength=len(edges) - 1)

#### Load the data
The fitted parameter vectors of `scripts/kin_vc_drift_extensions.py` (`data/kin_vc_drift_ext_vectors.npz`); the RR Lyrae
prediction (`results/kin_vc_drift_rrl.csv`) and the likelihood deficit per bin (`results/kin_vc_drift_deficit.csv`) from the same
script; the giant catalogues of Zenodo bundle 1 through `lmcold.samples.giants()`, split into the five [Fe/H] bins and cut to
$2<\rho<9^\circ$ by `ring`.

In [ ]:
z = np.load(DATA / 'kin_vc_drift_ext_vectors.npz')  # parameter vectors per bin and model ('free', 'drift', ...), and 'names'
names = list(z['names'])  # the giant bins, in the order of the vectors
rrl = pd.read_csv(ROOT / 'results/kin_vc_drift_rrl.csv', index_col=0)  # RR Lyrae v_rot(4 kpc), measured and predicted
Dd = pd.read_csv(ROOT / 'results/kin_vc_drift_deficit.csv')  # summed ln L(shared v_c + drift) - ln L(free) per bin of R, phi
# Giant catalogue dict per [Fe/H] bin (data/giants_apogee.parquet, giants_boss.parquet), 2 < rho < 9 deg, no distance moduli.
giants = [ring(cat | dict(dmu=np.full(len(cat['ra']), np.nan))) for _, cat in samples.giants()]

#### The fitted models
For each giant [Fe/H] bin the parameter vector of the free fit (its own rotation curve) and of the shared-$v_c$ + drift fit (one
circular-speed curve; the bin's mean rotation lags it by its asymmetric drift, `lmcold.kinfit.rotation` with $h>0$).

In [ ]:
plt.style.use(ROOT / 'src/lmcold/paper.mplstyle')
pf, pj = z['free'], z['drift']  # (5 bins, 33 parameters, order of lmcold.kinfit.PARAMS): free fit, shared-v_c + drift fit
labels = [n.replace('giants ', '') for n in names]

#### Rotation curves and $v_{\rm rot}(4\,{\rm kpc})$
(a) The shared $v_c(R)$ and each bin's mean rotation in both models; (b) $v_{\rm rot}(4\,{\rm kpc})$ of the shared-$v_c$ +
drift model against the free fit, with the RR Lyrae prediction (`results/kin_vc_drift_rrl.csv`).

In [ ]:
fig, ax = panel_grid(1, 2, width=6.0, left=0.55, right=0.1, hgap=0.55, top=0.75)
R = np.linspace(1.75, 9, 100)  # in-plane radius grid [kpc]
# Shared circular speed: the first 32 parameters with h (index 32) set to 0 switch the drift off (kinfit.rotation).
ax[0, 0].plot(R, np.asarray(kinfit.rotation(np.r_[pj[0][:32], 0.], R)), 'k-', lw=1.5, label=r'shared $v_c$')
# (a) Each bin's mean rotation [km/s]: free fit (solid) and shared v_c minus its drift (dashed).
for a, j, c, lab in zip(pf, pj, COLOURS, labels):
    ax[0, 0].plot(R, np.asarray(kinfit.rotation(a, R)), '-', color=c, lw=0.9)
    ax[0, 0].plot(R, np.asarray(kinfit.rotation(j, R)), '--', color=c, lw=0.9, label=lab)
ax[0, 0].set(xlabel='R [kpc]', ylabel=r'mean rotation [km s$^{-1}$]', ylim=(0, 110))
ax[0, 0].legend(loc='lower left', bbox_to_anchor=(0, 1.02), ncol=3, fontsize=5, frameon=False,
                title='solid: free; dashed: shared $v_c$ + drift', title_fontsize=5)
# (b) v_rot(4 kpc) [km/s] of the shared-v_c + drift fit against the free fit, one point per giant bin.
for a, j, c in zip(pf, pj, COLOURS):
    ax[0, 1].plot(vrot4(a), vrot4(j), 'o', color=c, ms=5)
# RR Lyrae: measured (x) and predicted (y) v_rot(4 kpc), median with 16-84 per cent errors; grey line is y = x.
m, pr = rrl['measured'], rrl['predicted']
ax[0, 1].errorbar(m['median'], pr['median'], xerr=[[m['median'] - m['p16']], [m['p84'] - m['median']]],
                  yerr=[[pr['median'] - pr['p16']], [pr['p84'] - pr['median']]], fmt='*', color='#7b3294', ms=8,
                  label='RR Lyrae (predicted)')
ax[0, 1].plot([50, 90], [50, 90], color='0.6', lw=0.6)
ax[0, 1].set(xlabel=r'free $v_{\rm rot}(4\,{\rm kpc})$ [km s$^{-1}$]', ylabel=r'shared $v_c$ + drift [km s$^{-1}$]',
             xlim=(50, 90), ylim=(50, 90))
ax[0, 1].legend(loc='lower left', bbox_to_anchor=(0, 1.02), fontsize=6, frameon=False)
# Panel letters, then save.
for a, t in zip(ax[0], 'ab'):
    a.text(0.04, 0.92, f'({t})', transform=a.transAxes)
fig.savefig(ROOT / 'figures/fig_kin_vc_drift.png')

In [ ]:
from IPython.display import Image
Image(filename='figures/fig_kin_vc_drift.png')

**Figure 7.8.** (a) The shared circular speed $v_c(R)$ (black) and the mean rotation $\bar v_\phi(R)$ of each giant [Fe/H] bin
(colours, legend above), free (solid) and in the shared-$v_c$ + drift model (dashed), against cylindrical radius $R$ (kpc).
(b) Rotation at $R$ = 4 kpc, $v_{\rm rot}(4\,{\rm kpc})$ (km/s), of the shared-$v_c$ + drift model against the free fit, one point per
bin; star: the RR Lyrae, with the rotation the model predicts for them (vertical bar) against the measured one (horizontal bar),
16th–84th percentiles; grey line: equality. `scripts/fig_kin_vc_drift.py`

#### Where the shared model fits worse
Summed $\ln L$ of the shared-$v_c$ + drift fit minus the free fit, star by star, in bins of radius (a) and azimuth (b)
(`results/kin_vc_drift_deficit.csv`).

In [ ]:
fig, ax = panel_grid(1, 2, width=6.0, left=0.6, right=0.1, hgap=0.6, top=0.5)
# One line per giant bin: summed ln L difference per R bin (a) and per azimuth bin (b), at the bin centres; the 'survey' rows
# of the table are not plotted.
for (name, g), c in zip(Dd[Dd.coordinate != 'survey'].groupby('population', sort=False), COLOURS):
    for a, coord in zip(ax[0], ('R', 'phi')):
        q = g[g.coordinate == coord]
        a.plot(.5 * (q.lo + q.hi), q.dlnL, 'o-', color=c, ms=3, lw=0.8, label=name.replace('giants ', ''))
# Zero line (both fits equally good), axis labels and panel letters.
for a, lab, t in zip(ax[0], ('R [kpc]', r'$\phi$ from the line of nodes [deg]'), 'ab'):
    a.axhline(0, color='0.6', lw=0.5)
    a.set(xlabel=lab, ylabel=r'$\sum\,\ln L_{\rm shared\,v_c+drift}-\ln L_{\rm free}$')
    a.text(0.04, 0.92, f'({t})', transform=a.transAxes)
ax[0, 0].legend(loc='lower left', bbox_to_anchor=(0, 1.02), ncol=3, fontsize=5, frameon=False)
fig.savefig(ROOT / 'figures/fig_kin_vc_drift_deficit.png')

In [ ]:
from IPython.display import Image
Image(filename='figures/fig_kin_vc_drift_deficit.png')

**Figure 7.9.** Difference of the log-likelihoods of the shared-$v_c$ + drift and the free fits, summed over the
stars of each giant [Fe/H] bin (colours) in bins of cylindrical radius $R$ (a; kpc) and of azimuth $\phi$ in the disc plane, measured
from the line of nodes along the rotation (b; deg). Negative: the shared model fits worse. `scripts/fig_kin_vc_drift.py` Comparing the binned data with the
models (Figure 7.10): in the most metal-poor bin the data at 3–5 kpc (61--64 km/s) follow the shared model, but beyond 7 kpc they stay flat
(54--56 km/s) where the model falls to 46--51 km/s, because its drift term $\sigma_R^2 R/h$ grows too fast; at 2–3 kpc both
models miss. Against azimuth (Figure 7.11) the two models leave almost the same residuals: the shared circular
speed changes the radial profile of the rotation, not its pattern around the disc.

#### Data against the models
The mean $v_\phi$ and $v_R$ of the giants with 3D velocities, in bins of radius and of azimuth, against the models averaged over
the same stars (module docstring); the binned values go to `results/kin_vc_drift_profiles.csv`.

In [ ]:
rows = []  # one dict per population x model x coordinate x bin; written to results/kin_vc_drift_profiles.csv
# Two figures of 2 x 5 panels: against R (figR) and against azimuth (figP); one column per giant bin.
figR, axR = panel_grid(2, 5, width=7.2, left=0.5, right=0.08, hgap=0.12, vgap=0.12, top=0.4)
figP, axP = panel_grid(2, 5, width=7.2, left=0.5, right=0.08, hgap=0.12, vgap=0.12, top=0.4)
for col, (cat, lab, c) in enumerate(zip(giants, labels, COLOURS)):
    # Stars with a 3D velocity (both a line-of-sight velocity and a proper motion), in the fit's data layout.
    has = np.isfinite(cat['vlos']) & np.isfinite(cat['pmra'])
    d = kinfit.data(*(cat[k][has] for k in COLS))
    # Measured and model internal velocities of these stars for each model in MODELS (velocities()).
    res = {key: velocities(z[key][col], d) for key in MODELS}
    keep = res['free'][2] < CLIP  # 3D residual from the free fit below CLIP [km/s], same stars for every model
    for key, ((ophi, oR), (mphi, mR), _, R, phi) in res.items():
        for coord, x, edges in (('R', R, R_EDGES), ('phi', phi, PHI_EDGES)):
            # Azimuth bins use only 2 < R < 9 kpc (the R_EDGES range).
            sel = keep & ((R > R_EDGES[0]) & (R < R_EDGES[-1]) if coord == 'phi' else keep)
            # Binned means of data and model v_phi, v_R [km/s]; err is the standard error of data v_phi, errR of data v_R.
            (dphi, dR, mp, mr), err, n = binned(x[sel], edges, ophi[sel], oR[sel], mphi[sel], mR[sel])
            errR = binned(x[sel], edges, oR[sel])[1]
            rows += [dict(population=names[col], model=key, coordinate=coord, lo=edges[i], hi=edges[i + 1], n=n[i],
                          data_vphi=dphi[i], model_vphi=mp[i], err_vphi=err[i], data_vR=dR[i], model_vR=mr[i], err_vR=errR[i])
                     for i in range(len(edges) - 1)]
    # The rows of this giant bin.
    T = pd.DataFrame(rows)
    T = T[T.population == names[col]]
    # v_phi against R.
    x = .5 * (R_EDGES[1:] + R_EDGES[:-1])  # R bin centres [kpc]
    # Data points in the free-fit frame (top row).
    t = T[(T.model == 'free') & (T.coordinate == 'R')]
    axR[0, col].errorbar(x, t.data_vphi, t.err_vphi, fmt='o', color=c, ms=3, lw=0.8, zorder=3, label='data')
    for key, style in (('free', dict(color='0.55', ls='-')), ('drift', dict(color='k', ls='--')),
                       ('drift_p_stream_p_beta', dict(color='k', ls=':'))):
        t = T[(T.model == key) & (T.coordinate == 'R')]
        axR[0, col].plot(x, t.model_vphi, lw=1, label=MODELS[key], **style)
        # Bottom row: data - model for free (open, shifted -0.1 kpc) and shared v_c + drift (filled, +0.1 kpc).
        if key != 'drift_p_stream_p_beta':
            axR[1, col].errorbar(x + (0.1 if key == 'drift' else -0.1), t.data_vphi - t.model_vphi, t.err_vphi,
                                 fmt='o', mfc=c if key == 'drift' else 'w', mec=c, ecolor=c, ms=3, lw=0.8, label=MODELS[key])
    # Residuals against azimuth.
    xp = .5 * (PHI_EDGES[1:] + PHI_EDGES[:-1])  # azimuth bin centres [deg]; points shifted by -4 (free) and +4 deg (drift)
    for key in ('free', 'drift'):
        t = T[(T.model == key) & (T.coordinate == 'phi')]
        for row, (dv, mv, ev) in enumerate((('data_vphi', 'model_vphi', 'err_vphi'), ('data_vR', 'model_vR', 'err_vR'))):
            axP[row, col].errorbar(xp + (4 if key == 'drift' else -4), t[dv] - t[mv], t[ev], fmt='o', mfc=c if key == 'drift' else 'w',
                                   mec=c, ecolor=c, ms=3, lw=0.8, label=MODELS[key])
    axR[0, col].set_title(lab, fontsize=6)
    axP[0, col].set_title(lab, fontsize=6)
# Zero lines on every residual panel.
for a in axR[1].tolist() + axP.ravel().tolist():
    a.axhline(0, color='0.6', lw=0.5)
for axs, xlab, ylabs, xlim in ((axR, 'R [kpc]', (r'mean $v_\phi$ [km s$^{-1}$]', r'data $-$ model $v_\phi$ [km s$^{-1}$]'), (1.6, 9.4)),
                               (axP, r'$\phi$ [deg]', (r'data $-$ model $v_\phi$ [km s$^{-1}$]', r'data $-$ model $v_R$ [km s$^{-1}$]'), (0, 360))):
    # Common y range per row (the widest of its panels); y labels on the first column, x labels on the bottom row.
    for row in range(2):
        lim = np.array([a.get_ylim() for a in axs[row]])
        for col, a in enumerate(axs[row]):
            a.set(xlim=xlim, ylim=(lim[:, 0].min(), lim[:, 1].max()))
            if col:
                a.tick_params(labelleft=False)
            else:
                a.set_ylabel(ylabs[row], fontsize=6)
            if row == 0:
                a.tick_params(labelbottom=False)
            else:
                a.set_xlabel(xlab)
# Azimuth ticks every 90 deg; legends above the first column (lower-left inside the bottom-left panel of figR).
for a in axP.ravel():
    a.set_xticks([0, 90, 180, 270])
axR[0, 0].legend(loc='lower left', bbox_to_anchor=(0, 1.15), ncol=4, fontsize=5, frameon=False)
axR[1, 0].legend(loc='lower left', fontsize=5, frameon=False)
axP[0, 0].legend(loc='lower left', bbox_to_anchor=(0, 1.15), ncol=2, fontsize=5, frameon=False)
# Save both figures and the binned profiles.
figR.savefig(ROOT / 'figures/fig_kin_vc_drift_data.png')
figP.savefig(ROOT / 'figures/fig_kin_vc_drift_data_phi.png')
pd.DataFrame(rows).to_csv(ROOT / 'results/kin_vc_drift_profiles.csv', index=False, float_format='%.3f')

In [ ]:
from IPython.display import Image
Image(filename='figures/fig_kin_vc_drift_data.png')

**Figure 7.10.** Data against the models, one column per giant [Fe/H] bin. Top: mean azimuthal velocity $v_\phi$
(km/s) of the giants with proper motions and line-of-sight velocities in 1-kpc bins of cylindrical radius $R$ (points, error: standard
deviation over $\sqrt n$ for $n$ stars), and of the models averaged over the same stars: free (grey), shared $v_c$ + drift (dashed),
and shared $v_c$ + drift with population-dependent streaming and thickness (dotted). Bottom: data minus model for the free (open) and
the shared-$v_c$ + drift (filled) fits. `scripts/fig_kin_vc_drift.py`

In [ ]:
from IPython.display import Image
Image(filename='figures/fig_kin_vc_drift_data_phi.png')

**Figure 7.11.** Data minus model of the mean azimuthal velocity $v_\phi$ (top) and radial velocity $v_R$
(bottom; km/s) of the same giants in 45° bins of azimuth $\phi$ (from the line of nodes, along the rotation) at 2 < $R$ < 9 kpc, for
the free (open) and the shared-$v_c$ + drift (filled) fits; one column per [Fe/H] bin. `scripts/fig_kin_vc_drift.py`

## Where one circular speed is not enough

The shared model fits the giants worse than free rotation curves: the log-likelihood $\ln L$ of the data is lower by
$\Delta\ln L$ = 147, for 12 fewer parameters. The deficit is spread over all radii and all five bins (25 to 39 per bin),
not one region or survey (Figure 7.9).

The deficit is not caused by the scale lengths, the tracers or the streaming:

- with each $h$ at the 16th or the 84th percentile of its bootstrap, $\Delta\ln L$ = 133 and 166;
- on the 49 457 giants with SDSS-V BOSS spectra that are also in the mass sample, with $h$ measured from those stars, 112;
- population-dependent bisymmetric ($m=2$) streaming improves the free model by 89 and leaves the gap at 165;
- letting the circular speed a population feels fall with its vertical dispersion (a thick population feels a weaker average radial
  force) gives $v_c(4\,{\rm kpc})$ = 82.7 km/s for the most metal-poor to 88.5 km/s for the most metal-rich bin and
  $\Delta\ln L$ = 113 (38 with streaming as well).

Left free, the scale lengths fall to 0.4–1.0 kpc, about half those measured (0.9–1.9 kpc), and
$v_c(4\,{\rm kpc})$ rises to 101.3 km/s. One potential with the drift of a thin, steady disc therefore describes the old
population's rotation at 3–5 kpc but not over the whole disc: the old population is thick and may not be fully in equilibrium. The
circular speed this model gives depends on these assumptions, which is why the Letter does not use it.

### The diagnostic fits

`scripts/kin_vc_drift_extensions.py` makes the fits of this section. It refits the free and the shared-$v_c$ + drift models on all
giants, and the latter again with every $h$ at the 16th and at the 84th percentile of its bootstrap; it sums the difference of the
per-star log-likelihoods of the two models in bins of $R$ and $\phi$ and per survey (Figure 7.9); it repeats
both fits on the BOSS giants that are also in the mass sample, with $h$ measured from those stars; and it adds the
population-dependent streaming and the thickness term, alone and together. It writes `results/kin_vc_drift_ext.csv`,
`results/kin_vc_drift_deficit.csv` and the fitted parameter vectors (`data/kin_vc_drift_ext_vectors.npz`, Zenodo bundle 1), from
which `scripts/fig_kin_vc_drift.py` draws the figures of this page.

<details><summary>About <code>scripts/kin_vc_drift_extensions.py</code></summary>

```text
Why does one circular speed plus asymmetric drift fit the giant bins worse than free rotation curves (scripts/kin_vc_drift.py)?
Diagnostics and extensions, in this order:

1. Where the deficit comes from: per-star ln L of the 'drift' fit minus the 'free' fit (both joint ML, same stars), summed in bins
   of in-plane radius R and azimuth phi (from the line of nodes, along e_phi) per giant bin, and per survey (APOGEE, BOSS).
2. Matched tracers: the BOSS kinematic giants that are also in the BOSS mass sample (lmcold.mass_samples.samples: the same
   stars, velocity window 150-420 km/s), with h measured from those very stars (selection weights), 'free' and 'drift' refitted;
   and, on all giants, 'drift' with h at the 16th and 84th percentiles of its bootstrap (kin_vc_drift_h.csv) for every bin.
3. Population-dependent streaming: m2c_g = m2c + a_c q_g and m2s_g = m2s + a_s q_g, q_g = Q (a fixed ordering from metal-poor to
   metal-rich, centred), two shared parameters, in 'free + stream' and 'drift + stream'.
4. Thickness (diagnostic): each bin feels an effective circular speed v_c,g^2 = v_c^2 (1 - beta (sigma_z,g / SIGMA_REF)^2), with
   sigma_z,g its own fitted vertical dispersion and beta one shared parameter ('drift + beta', 'drift + stream + beta'): a thick
   population feels a weaker vertically averaged radial force. A phenomenological test, not a vertical-force integration.

All fits: scripts/kin_vc_drift.joint_fit (geometry, systemic motion and streaming shared and free; CURVE shared for the drift models;
OWN per bin; h fixed). Writes results/kin_vc_drift_ext.csv (variant, population, n, nll, dlnL against 'free' of the same stars, vrot4,
vc4 (the bin's effective v_c at 4 kpc), sigma_R, sigma_phi, sigma_z, dlnsig_dR, k, h, extra parameters, success, grad),
results/kin_vc_drift_deficit.csv (population, coordinate, bin, n, dlnL) (figures: scripts/fig_kin_vc_drift.py); fitted vectors in data/kin_vc_drift_ext_vectors.npz.

The numbers of the KB page 'Rotation and the circular speed' (section 'Where one circular speed is not enough') are written from
these tables by scripts/paper_text_vc_drift.py. Diagnostic (not in the Letter).

Inputs: the release giants (lmcold.samples.giants(), Zenodo bundle 1); results/kin_vc_drift_h.csv (scripts/kin_vc_drift.py); the
joint ellipsoid fit results/kin_mcmc_data_ellipsoid.csv (through kin_vc_drift.joint_params); the BOSS mass sample
(lmcold.mass_samples.samples) with data/giant_foreground.parquet.

Run: .venv/bin/python scripts/kin_vc_drift_extensions.py   (after scripts/kin_vc_drift.py; about 5 h in the run of 2026-11-09)
```
</details>

*Shown, not run here: joint maximum-likelihood fits of the giant bins in every model variant; about 5 h. The cells below are the script itself; run it from the repository with* `python scripts/kin_vc_drift_extensions.py`*.*

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

#### Settings
The ordering of the bins used by the population-dependent streaming, the reference dispersion and bounds of the thickness term,
and the bins of radius and azimuth of the deficit map.

```python
import healpy as hp
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

from fit_kinematics_mcmc import COLS, ring
from kin_vc_drift import CURVE, P, SHARED_FREE, joint_fit, joint_params, scale_length, radius, vrot4, with_h
from lmcold import geometry as geo
from lmcold import kinfit, samples
from lmcold.mass_samples import samples as mass_samples
from lmcold.sources import DATA, ROOT

Q = np.array([-1.0, -0.5, 0.0, 0.5, 1.0])  # streaming ordering of the giant bins, metal-poor to metal-rich
SIGMA_REF = 30.0  # km/s, reference vertical dispersion of the thickness term
# Extra shared parameters of each extension: (names, starting values, bounds). a_c, a_s: slopes of m2c, m2s against Q; beta: the
# thickness coefficient (dimensionless).
STREAM_ADD = (('a_c', 'a_s'), (0.0, 0.0), ((-0.5, 0.5), (-0.5, 0.5)))
BETA_ADD = (('beta',), (0.0,), ((-0.5, 0.75),))  # beta < 0.75 keeps 1 - beta (sigma_z / 30)^2 > 0 for sigma_z < 34 km/s
R_BINS, PHI_BINS = np.arange(1., 11., 1.), np.linspace(0, 360, 9)  # kpc (1-kpc bins), deg (45-deg bins)
```

#### Helpers
The two extensions as functions that modify the population vectors inside a joint fit, the 'drift' fit started from the free
solution, the output rows, and the in-plane azimuth of each star.

```python
def adjust_stream(ps, v, offset=0):
    """Per-bin m = 2 coefficients: shared value plus slope times Q.

    ps : list of five kinfit.PARAMS vectors (jax arrays); v : the extra parameters of the fit, of which v[offset], v[offset + 1] are
    a_c, a_s. Returns the modified vectors.
    """
    # Bin i: m2c + a_c Q[i] and m2s + a_s Q[i] (jax functional updates).
    return [p.at[P('m2c')].add(v[offset] * Q[i]).at[P('m2s')].add(v[offset + 1] * Q[i]) for i, p in enumerate(ps)]


def adjust_beta(ps, v, offset=0):
    """Effective circular speed per bin: v0 scaled by sqrt(1 - beta (sigma_z / SIGMA_REF)^2), the square of the scale clipped at
    1e-3 from below.

    ps : list of five kinfit.PARAMS vectors (jax arrays); v : the extra parameters of the fit, v[offset] = beta. Returns the
    modified vectors.
    """
    # Each bin's v0 times sqrt(max(1 - beta (sigma_z / SIGMA_REF)^2, 1e-3)), sigma_z being that bin's own fitted value.
    return [p.at[P('v0')].multiply(kinfit.jnp.sqrt(kinfit.jnp.clip(1 - v[offset] * (p[P('sigma_z')] / SIGMA_REF) ** 2, 1e-3)))
            for p in ps]


# Extension name: the `add` argument of kin_vc_drift.joint_fit (names, starts, bounds, function applying them).
ADD = {'stream': (STREAM_ADD[0], STREAM_ADD[1], STREAM_ADD[2], adjust_stream),
       'beta': (BETA_ADD[0], BETA_ADD[1], BETA_ADD[2], adjust_beta),
       'stream + beta': (STREAM_ADD[0] + BETA_ADD[0], STREAM_ADD[1] + BETA_ADD[1], STREAM_ADD[2] + BETA_ADD[2],
                         lambda ps, v: adjust_beta(adjust_stream(ps, v, 0), v, 2))}


def fit_drift(ds, pfree, hs, add=None):
    """'drift' joint ML from the free solution pfree with scale lengths hs: the shared curve alone first (from there the first
    L-BFGS-B step of the full problem can reach the corners of BOUNDS), then everything."""
    # Start: the free vectors with h set and the shared curve (v0 [km/s], R0 [kpc], vslope [1/kpc]) = (90, 2.5, 0) in every bin.
    p0 = with_h(pfree, hs)
    for p in p0:
        p[[P(n) for n in CURVE]] = [90., 2.5, 0.]
    # Fit the curve alone, then everything free in 'drift' (SHARED_FREE + CURVE), with the extension `add` if given.
    p0, _, _, _ = joint_fit(ds, p0, CURVE, own=())
    return joint_fit(ds, p0, SHARED_FREE + CURVE, add=add)


def rows(variant, ps, nll, info, names, ds, ref_nll, add_names=()):
    """Output rows of one fit (one per bin), with dlnL = ref_nll - nll against the 'free' fit of the same stars; prints a summary.

    variant : name of the fit; ps : fitted vectors; nll : -lnL; info : the info dict of joint_fit; names, ds : bin labels and data;
    add_names : names of the extra parameters (info['extra']). vrot4 and vc4 in km/s, at R = 4 kpc.
    """
    # One row per bin: counts, likelihoods, rotation at 4 kpc, dispersions, scale length, streaming, inclination, extra parameters.
    out = []
    for name, p, d in zip(names, ps, ds):
        # vc4: vrot4 with h (index 32 of kinfit.PARAMS) set to 0, i.e. the circular speed without drift (kin_vc_drift.vrot4).
        r = dict(variant=variant, population=name, n=len(np.asarray(d['pmra'])), nll=nll, dlnL=ref_nll - nll, vrot4=vrot4(p),
                 vc4=vrot4(np.r_[p[:32], 0.0]), success=info['success'], grad=info['grad'])
        r |= {n: p[P(n)] for n in ('sigma_R', 'sigma_phi', 'sigma_z', 'dlnsig_dR', 'k', 'h', 'm2c', 'm2s', 'inc')}
        r |= dict(zip(add_names, info['extra']))
        out.append(r)
    print(variant, f'nll {nll:.1f}, dlnL vs free {ref_nll - nll:+.1f}', 'vrot4', [round(x['vrot4'], 1) for x in out],
          'vc4', [round(x['vc4'], 1) for x in out], 'extra', np.round(info['extra'], 3), 'ok' if info['success'] else 'NOT converged',
          f"grad {info['grad']:.2g}", flush=True)
    return out


def azimuth(p, d):
    """In-plane radius [kpc] and azimuth phi [deg] (from the line of nodes, increasing along e_phi) of the stars of d."""
    # R [kpc] and in-plane unit vector e_R (N, 3) of each star from the model; line of nodes and normal of the fitted orientation.
    _, _, _, _, R, eR, _, n = (np.asarray(v) for v in kinfit._model(p, d['rhat'], d['ea'], d['ed']))
    _, _, _, line, nn = (np.asarray(v) for v in geo.symmetry_axis(*(float(v) for v in kinfit.centre(p[0], p[1])),
                                                                     np.deg2rad(p[P('inc')]), np.deg2rad(p[P('theta')])))
    # phi = atan2(e_R . (n x line), e_R . line), in [0, 360) deg.
    return R, np.rad2deg(np.arctan2(eR @ np.cross(nn, line), eR @ line)) % 360
```

#### Load the data
The disc frame of the joint ellipsoid fit, the giants of the five [Fe/H] bins at 2 < $\rho$ < 9 deg and the scale lengths of
`results/kin_vc_drift_h.csv`. The BOSS mass sample and the foreground table are read in the matched-tracer step.

```python
frames = joint_params()  # population -> kinfit.PARAMS vector (results/kin_mcmc_data_ellipsoid.csv, stage 2)
giants = [(f'giants {lab}', ring(cat | dict(dmu=np.full(len(cat['ra']), np.nan)))) for lab, cat in samples.giants()]
# Bin labels and the kinfit data of each bin (columns COLS of scripts/fit_kinematics_mcmc.py; distances NaN, not used).
names = [n for n, _ in giants]
ds = [kinfit.data(*(c[k] for k in COLS)) for _, c in giants]
H = pd.read_csv(ROOT / 'results/kin_vc_drift_h.csv').set_index('population')  # h, h_p16, h_p84 [kpc] per population
out, vectors = [], {}  # output rows; fitted vectors per variant (saved to data/kin_vc_drift_ext_vectors.npz)
```

#### Reference fits and the scale-length bracket
'free' and 'drift' on all giants, and 'drift' with every $h$ at the 16th and at the 84th percentile of its bootstrap.

```python
# 'free': own rotation curve per bin, no drift (h = 0); its -lnL nllf is the reference of every dlnL on all giants.
pf, _, nllf, inf_f = joint_fit(ds, with_h([frames[n] for n in names], [0.] * 5), SHARED_FREE)
# Rows of the 'free' fit (dlnL = 0 by definition); fitted vectors kept for the npz file.
out += rows('free', pf, nllf, inf_f, names, ds, nllf)
# 'drift' with the measured h, then with every h at its 16th and at its 84th bootstrap percentile.
for variant, col in (('drift', 'h'), ('drift, h p16', 'h_p16'), ('drift, h p84', 'h_p84')):
    pj, _, nllj, inf_j = fit_drift(ds, pf, H.loc[names, col].to_numpy())
    out += rows(variant, pj, nllj, inf_j, names, ds, nllf)
    vectors[variant] = np.array(pj)
vectors['free'] = np.array(pf)
```

#### Where the deficit comes from
The per-star log-likelihood of 'drift' minus that of 'free', summed in bins of $R$ and of azimuth $\phi$ and per survey, for
each giant bin. Writes `results/kin_vc_drift_deficit.csv`.

```python
deficit = []
for name, (_, c), d, a, j in zip(names, giants, ds, pf, vectors['drift']):
    # Per-star ln L of 'drift' (j) minus 'free' (a), ellipsoid likelihood; negative where 'drift' fits worse.
    dl = np.asarray(kinfit.loglike_stars(j, d, True)) - np.asarray(kinfit.loglike_stars(a, d, True))
    R, phi = azimuth(j, d)
    # Sum of dl in 1-kpc bins of R and 45-deg bins of phi (stars outside the edges fall in no bin).
    for coord, x, edges in (('R', R, R_BINS), ('phi', phi, PHI_BINS)):
        k = np.digitize(x, edges) - 1
        for i in range(len(edges) - 1):
            deficit.append(dict(population=name, coordinate=coord, lo=edges[i], hi=edges[i + 1], n=int((k == i).sum()),
                                dlnL=float(dl[k == i].sum())))
    # Sum of dl per survey.
    for sv in ('apogee', 'boss'):
        m = np.asarray(c['survey']) == sv
        deficit.append(dict(population=name, coordinate='survey', lo=np.nan, hi=np.nan, n=int(m.sum()), dlnL=float(dl[m].sum()),
                            survey=sv))
# Write the deficit table and print the per-survey sums.
D = pd.DataFrame(deficit)
D.to_csv(ROOT / 'results/kin_vc_drift_deficit.csv', index=False, float_format='%.3f')
print(D[D.coordinate == 'survey'][['population', 'survey', 'n', 'dlnL']].round(1).to_string(index=False), flush=True)
```

#### Matched tracers
The BOSS kinematic giants that are also in the BOSS mass sample (matched within 1 arcsec), with $h$ measured from those same
stars, refitted with 'free' and 'drift'.

```python
# The BOSS mass sample (ra, dec [deg], selection weight w), joined to the foreground table (coverage fill, Milky Way fraction
# f_mw).
b, _, _ = mass_samples()
fg = pd.read_parquet(DATA / 'giant_foreground.parquet')
b = b.merge(fg[['source_id', 'fill', 'f_mw']], on='source_id', validate='one_to_one')
b['W'] = b.w * b['fill'] * (1 - b.f_mw)  # weight: selection x coverage fill x (1 - Milky Way fraction), as kin_vc_drift.py
unit = lambda ra, dec: np.c_[np.cos(np.deg2rad(dec)) * np.cos(np.deg2rad(ra)), np.cos(np.deg2rad(dec)) * np.sin(np.deg2rad(ra)),  # noqa: E731
                             np.sin(np.deg2rad(dec))]
# Unit vectors on the sky; a chord of 5e-6 is ~1 arcsec (4.85e-6 rad).
tree = cKDTree(unit(b.ra.to_numpy(), b.dec.to_numpy()))
mds, mhs, mframes = [], [], []
for name, c in giants:
    # Nearest mass-sample star of each kinematic giant; BOSS giants within 1 arcsec count as matched.
    dist, idx = tree.query(unit(c['ra'], c['dec']))
    m = (np.asarray(c['survey']) == 'boss') & (dist < 5e-6)  # within 1 arcsec
    # Matched kinematic giants (cm) and their mass-sample rows (s); h from those stars' weighted surface density, bootstrapped
    # over
    # HEALPix pixels (nside 64) as in kin_vc_drift.scale_length.
    cm = {k: np.asarray(v)[m] for k, v in c.items()}
    s = b.iloc[idx[m]]
    Rm = radius(frames[name], s.ra.to_numpy(), s.dec.to_numpy())
    mhs.append(scale_length(Rm, s.W.to_numpy(), hp.ang2pix(64, s.ra.to_numpy(), s.dec.to_numpy(), lonlat=True))[0])
    mds.append(kinfit.data(*(cm[k] for k in COLS)))
    print(f'{name}: matched BOSS {m.sum()} of {len(m)}, h {mhs[-1]:.2f} kpc', flush=True)
# 'free' and 'drift' on the matched stars; dlnL against the matched 'free' fit.
pmf, _, nllmf, inf_mf = joint_fit(mds, with_h([frames[n] for n in names], [0.] * 5), SHARED_FREE)
out += rows('matched: free', pmf, nllmf, inf_mf, names, mds, nllmf)
# 'drift' on the matched stars with their own h; both fits' vectors kept.
pmj, _, nllmj, inf_mj = fit_drift(mds, pmf, mhs)
out += rows('matched: drift', pmj, nllmj, inf_mj, names, mds, nllmf)
vectors['matched: free'], vectors['matched: drift'] = np.array(pmf), np.array(pmj)
```

#### Population-dependent streaming and thickness
On all giants with the measured $h$: 'free + stream', and 'drift' with population-dependent streaming, with the thickness
term $\beta$, and with both. Writes `results/kin_vc_drift_ext.csv` and the fitted vectors.

```python
hs = H.loc[names, 'h'].to_numpy()
# 'free + stream' from the 'free' solution; then 'drift' with each extension (stream, beta, both), all against 'free'.
pfs, _, nllfs, inf_fs = joint_fit(ds, pf, SHARED_FREE, add=ADD['stream'])
out += rows('free + stream', pfs, nllfs, inf_fs, names, ds, nllf, ADD['stream'][0])
vectors['free + stream'] = np.array(pfs)
for key in ('stream', 'beta', 'stream + beta'):
    pjx, _, nllx, infx = fit_drift(ds, pf, hs, add=ADD[key])
    out += rows(f'drift + {key}', pjx, nllx, infx, names, ds, nllf, ADD[key][0])
    vectors[f'drift + {key}'] = np.array(pjx)
T = pd.DataFrame(out)
T.to_csv(ROOT / 'results/kin_vc_drift_ext.csv', index=False, float_format='%.4f')
# Fitted vectors per variant, keys made valid names (spaces to _, ':' and ',' dropped, '+' to p).
np.savez(DATA / 'kin_vc_drift_ext_vectors.npz', names=np.array(names), **{k.replace(' ', '_').replace(':', '').replace('+', 'p').replace(',', ''): v
                                                                     for k, v in vectors.items()})
```